# 03 — Four-Category Healthy Purchasing: Repeated Switching and Variance Decomposition

**Purpose.** Document two pieces of model-free evidence from the 2008–2017 trip panel: (i) how much of the annual household healthy-food-spending-share variation is between versus within households, and (ii) three illustrative household trajectories with repeated crossings of a common descriptive threshold.

**Four healthy spending components:** fresh fruits and vegetables, **low-sugar cereal**, healthy snacks, and healthier beverages. Yogurt and milk are excluded. The cereal measure remains *low-sugar cereal only* (merging low and medium sugar in a separate HMM profile chart is not silently imposed here).

**Annual outcome:** `Y_it = annual spending in these four healthy groups / annual total food_spend`, aggregated across all observed trips. It is a share of the full food wallet, **not** a within-category composition share. No HMM posterior states or life-event coefficients enter this notebook.

**Variance decomposition:** All valid household-years with at least 12 trips contribute. The observation-weighted between/within sums of squares add exactly to the total, including for an unbalanced panel. Also report each calendar year's cross-sectional variance separately; those annual variances should **not** be added to the between/within shares.

**Repeated-switch illustrations:** The original notebook's purposive selection is retained: at least 8 consecutive valid years, 3 robust regime crossings, a pooled median threshold, and a ±2.5 percentage-point exclusion band. Household illustrations are **not** representative prevalence estimates.

**Privacy:** All CSVs, Parquet files, detailed household outputs, figures, and figure metadata are saved under the private local data directory, not to GitHub. Share outputs externally only after checking data permissions.


In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D

# Local data paths; never default research outputs to the Git repository.
# Set HEALTHY_PURCHASE_DATA_ROOT in the environment when working on another computer.
DATA_ROOT = Path(os.environ.get(
    "HEALTHY_PURCHASE_DATA_ROOT",
    r"C:\Users\rwen13\Documents\healthy_purchase_data"
))
SOURCE = DATA_ROOT / "processed" / "hmm_input" / "hmm_estimation_panel.parquet"
OUTPUT_DIR = DATA_ROOT / "model_outputs" / "model_free_evidence" / "switching_trajectories_four_category"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START, YEAR_END = 2008, 2017
MIN_ANNUAL_TRIPS = 12
N_EXAMPLES = 3
MIN_CONSECUTIVE_YEARS = 8
MIN_SWITCHES = 3
EXCLUSION_BAND_PP = 2.5
RANDOM_SEED = 20261031
DUCKDB_THREADS = min(4, max(1, (os.cpu_count() or 2) // 2))

HEALTHY_SPENDING_COLUMNS = [
    "fresh_spending",             # Fresh fruits and vegetables
    "low_sugar_cereal_spending",  # Low-sugar cereal only
    "healthy_snack_spending",     # Healthy snacks
    "healthy_beverage_spending",  # Healthier beverages
]
CACHE = OUTPUT_DIR / "annual_four_category_spending.parquet"
CACHE_META = OUTPUT_DIR / "annual_four_category_spending.meta.json"
assert MIN_SWITCHES >= 3 and EXCLUSION_BAND_PP > 0


## 1. Aggregate trip data to the household-year level

Use DuckDB to scan only the columns needed for the four-category annual spending share. The source is a trip-level panel; household IDs are treated as strings. A versioned cache avoids repeated scans of the large Parquet file. `food_spend` is **not** recalculated after removing milk/yogurt: it remains total food spending.

In [ ]:
if not SOURCE.is_file():
    raise FileNotFoundError(f"Four-category input HMM trip panel not found: {SOURCE}")

cache_metadata = {
    "source": str(SOURCE.resolve()),
    "size": SOURCE.stat().st_size,
    "mtime_ns": SOURCE.stat().st_mtime_ns,
    "columns": HEALTHY_SPENDING_COLUMNS,
    "years": [YEAR_START, YEAR_END],
    "aggregation_version": "four-category-annual-wallet-share-v2",
}

if (CACHE.is_file() and CACHE_META.is_file()
        and json.loads(CACHE_META.read_text(encoding="utf-8")) == cache_metadata):
    annual = pd.read_parquet(CACHE)
    print(f"Loaded cached annual dataset: {len(annual):,} household-years")
else:
    con = duckdb.connect()
    try:
        con.execute(f"SET threads={DUCKDB_THREADS}")
        con.execute("SET memory_limit='2GB'")
        temp_dir = OUTPUT_DIR / "duckdb_temp"
        temp_dir.mkdir(parents=True, exist_ok=True)
        con.execute(f"SET temp_directory='{temp_dir.as_posix()}'")
        escaped_source = SOURCE.as_posix().replace("'", "''")
        con.execute(f"CREATE VIEW trips AS SELECT * FROM read_parquet('{escaped_source}')")
        available = {r[0] for r in con.execute("DESCRIBE trips").fetchall()}
        required = {"hhid", "panel_year", "food_spend", *HEALTHY_SPENDING_COLUMNS}
        missing = required - available
        if missing:
            raise ValueError(f"Missing required panel columns: {sorted(missing)}")

        # COALESCE handles absent category purchases. Never treat missing food_spend as zero.
        expr = " + ".join(f'COALESCE("{c}", 0)' for c in HEALTHY_SPENDING_COLUMNS)
        annual = con.execute(f"""
            SELECT CAST(hhid AS VARCHAR) AS hhid,
                   CAST(panel_year AS INTEGER) AS panel_year,
                   COUNT(*) AS n_trips,
                   SUM(food_spend) AS food_spend,
                   SUM({expr}) AS healthy_spend,
                   COUNT(food_spend) AS n_nonmissing_food_spend
            FROM trips
            WHERE panel_year BETWEEN ? AND ?
            GROUP BY 1, 2
        """, [YEAR_START, YEAR_END]).fetchdf()
    finally:
        con.close()

    if annual.duplicated(["hhid", "panel_year"]).any():
        raise ValueError("Aggregation resulted in duplicate household-years")
    if (annual["healthy_spend"] < -1e-8).any():
        raise ValueError("Negative healthy spending detected; check preprocessing")
    # A household-year with missing trip food spending is not a reliable ratio.
    valid_denominator = (annual.food_spend > 0) & (
        annual.n_nonmissing_food_spend == annual.n_trips)
    annual["Y"] = (annual.healthy_spend / annual.food_spend.where(valid_denominator))
    invalid = annual.Y.notna() & ~annual.Y.between(-1e-8, 1+1e-8)
    if invalid.any():
        raise ValueError(f"Healthy share is outside [0, 1] for {int(invalid.sum()):,} household-years")
    annual.to_parquet(CACHE, index=False)
    CACHE_META.write_text(json.dumps(cache_metadata, indent=2), encoding="utf-8")
    print(f"Created annual cache: {len(annual):,} household-years")

if annual.duplicated(["hhid", "panel_year"]).any():
    raise ValueError("Cached annual data contain duplicate household-year keys")
print(f"Households: {annual.hhid.nunique():,}; years: {annual.panel_year.min()}–{annual.panel_year.max()}")


## 2. Define the eligible descriptive sample

Every eligible household-year must have at least `MIN_ANNUAL_TRIPS` observed trips, positive and complete food-spend denominators, and a healthy share within [0, 1]. **Variance decomposition uses every valid observation**, including short spells and one-year households. A second sensitivity analysis restricts to households with at least eight **consecutive** valid years.


In [ ]:
valid = annual.loc[
    annual.Y.notna() & annual.Y.between(0, 1)
    & (annual.n_trips >= MIN_ANNUAL_TRIPS),
    ["hhid", "panel_year", "n_trips", "Y"]
].sort_values(["hhid", "panel_year"]).copy()

if valid.empty:
    raise ValueError("No household-years satisfy the annual validity criteria")

# A missing or invalid year starts a new spell; it is never bridged.
new_spell = valid.hhid.ne(valid.hhid.shift()) | valid.panel_year.diff().ne(1)
valid["spell_id"] = new_spell.cumsum().astype("int64")
print(f"Variance sample: {len(valid):,} household-years, {valid.hhid.nunique():,} households")


## 3. Exact between/within household variance decomposition

Let `Y_it` be the annual healthier-food-spending share, `T_i` the number of eligible years for household `i`, and `mean_i` its observed-year mean. The overall mean is observation weighted. The identity is

\[
\sum_{it}(Y_{it}-ar Y)^2 = \sum_i T_i(ar Y_i-ar Y)^2 + \sum_{it}(Y_{it}-ar Y_i)^2.
\]

The **shares** are sums of squares divided by total sum of squares, not ratios of separately computed sample variances with different degrees-of-freedom corrections. For reference, `variance` columns divide each component by the total count of household-year observations (`N`), and thus add to the overall population variance. Values in squared-share units (e.g., 0.01 = 1 percentage-point squared).

In [ ]:
def decompose_between_within(frame: pd.DataFrame) -> pd.DataFrame:
    """Exactly partition unweighted household-year outcome variation.

    Each observed household-year contributes equally. Household means are based only on
    eligible observed years, so an unbalanced panel is handled without imputation.
    """
    if frame.empty or frame.Y.isna().any():
        raise ValueError("Decomposition input must be nonempty with no missing Y")
    household_mean = frame.groupby("hhid")["Y"].transform("mean")
    overall_mean = float(frame.Y.mean())
    between_ss = float(np.square(household_mean - overall_mean).sum())
    within_ss = float(np.square(frame.Y - household_mean).sum())
    total_ss = float(np.square(frame.Y - overall_mean).sum())
    if not np.isclose(between_ss + within_ss, total_ss, rtol=1e-10, atol=1e-12):
        raise AssertionError("Between + within must equal total sum of squares")
    if total_ss <= 0:
        raise ValueError("Zero total variation: percentage decomposition is undefined")
    n_obs = len(frame)
    return pd.DataFrame([
        {"component": "Between households", "sum_of_squares": between_ss,
         "variance": between_ss / n_obs, "share_percent": 100 * between_ss / total_ss},
        {"component": "Within households across years", "sum_of_squares": within_ss,
         "variance": within_ss / n_obs, "share_percent": 100 * within_ss / total_ss},
        {"component": "Total", "sum_of_squares": total_ss,
         "variance": total_ss / n_obs, "share_percent": 100.0},
    ]).assign(n_household_years=n_obs, n_households=frame.hhid.nunique(),
             overall_mean_share=overall_mean)

variance_decomposition = decompose_between_within(valid)
variance_decomposition.to_csv(OUTPUT_DIR / "variance_decomposition_four_category.csv", index=False)
display(variance_decomposition.round(6))

# Extra sensitivity check: restrict to households with a consecutive >=8-year valid spell.
spell_length = valid.groupby("spell_id").size()
long_spell_ids = spell_length.index[spell_length >= MIN_CONSECUTIVE_YEARS]
robust_households = valid.loc[valid.spell_id.isin(long_spell_ids), "hhid"].unique()
robust_valid = valid.loc[valid.hhid.isin(robust_households)].copy()
if not robust_valid.empty:
    variance_robustness = decompose_between_within(robust_valid)
    variance_robustness.to_csv(OUTPUT_DIR / "variance_decomposition_long_panel_robustness.csv", index=False)
    print("Robustness: households with >=8 consecutive eligible years")
    display(variance_robustness.round(6))
else:
    print("No households meet the >=8 consecutive eligible years robustness condition")


## 4. Year-by-year cross-sectional variance

Compute the variance **across different households within each calendar year**. Report both population variance (`ddof=0`) and sample variance (`ddof=1`), along with household count and annual mean. The descriptive 2008–2017 average is a simple average of the year-specific population variances; it is **not** a replacement for the exact two-way between/within decomposition above.

In [ ]:
yearly_cross_section = (
    valid.groupby("panel_year", sort=True)
    .agg(n_households=("hhid", "nunique"),
         mean_share=("Y", "mean"),
         variance_population=("Y", lambda s: float(s.var(ddof=0))),
         variance_sample=("Y", lambda s: float(s.var(ddof=1)) if len(s) > 1 else np.nan),
         std_population=("Y", lambda s: float(s.std(ddof=0))))
    .reset_index()
)
yearly_cross_section["mean_share_percent"] = yearly_cross_section.mean_share * 100
yearly_cross_section["std_percentage_points"] = yearly_cross_section.std_population * 100
# Convert from share-squared units to (percentage points)^2 for interpretation.
yearly_cross_section["variance_pp_squared"] = yearly_cross_section.variance_population * 10000
yearly_cross_section.to_csv(OUTPUT_DIR / "yearly_cross_sectional_variance_four_category.csv", index=False)
display(yearly_cross_section.round(5))
print("Mean annual cross-sectional population variance (share-squared):",
      round(float(yearly_cross_section.variance_population.mean()), 6))


## 5. Create and export variance figures

**Figure A:** One 100% stacked horizontal bar with exact Between/Within total-variation percentages. **Figure B:** The annual cross-sectional variance for 2008–2017, in squared percentage-point units. These figures visualize distinct statistics and should be labeled accordingly.

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42,
})
BLUE, ORANGE = "#2166AC", "#E58B45"
shares = variance_decomposition.set_index("component").loc[
    ["Between households", "Within households across years"], "share_percent"
].to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(11.5, 3.2))
left = 0.0
for fraction, label, color in zip(shares, ["Between households", "Within households"], [BLUE, ORANGE]):
    ax.barh(0, fraction, left=left, height=0.52, color=color, edgecolor="white", linewidth=2, label=label)
    if fraction >= 8:
        ax.text(left + fraction / 2, 0, f"{fraction:.1f}%", color="white", ha="center",
                va="center", fontsize=17, fontweight="bold")
    else:
        ax.annotate(f"{fraction:.1f}%", (left + fraction / 2, 0), xytext=(0, 20),
                    textcoords="offset points", ha="center", fontsize=10, color=color)
    left += fraction
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xticks([0, 20, 40, 60, 80, 100], ["0%", "20%", "40%", "60%", "80%", "100%"])
ax.grid(axis="x", alpha=0.12)
ax.set_axisbelow(True)
ax.set_title("Annual Healthy Purchasing: Variance Decomposition", fontsize=17, fontweight="bold", pad=15)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)
fig.text(.06, .03,
         f"Four categories | {len(valid):,} household-years | {valid.hhid.nunique():,} households | "
         f"{YEAR_START}–{YEAR_END} | Equal weight per household-year", fontsize=9, color="#555555")
fig.subplots_adjust(left=.06, right=.98, top=.75, bottom=.32)
for ext in ("png", "pdf"):
    fig.savefig(OUTPUT_DIR / f"variance_decomposition_four_category.{ext}", dpi=300, bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 4.5))
ax2.bar(yearly_cross_section.panel_year.astype(str), yearly_cross_section.variance_pp_squared,
        color=BLUE, alpha=0.85)
ax2.set_xlabel("Calendar year")
ax2.set_ylabel("Variance of annual healthy share (pp²)")
ax2.set_title("Cross-Sectional Variance Across Households Within Each Year", fontsize=14,
              fontweight="bold", pad=14)
ax2.grid(axis="y", alpha=.18)
ax2.set_axisbelow(True)
fig2.tight_layout()
for ext in ("png", "pdf"):
    fig2.savefig(OUTPUT_DIR / f"annual_cross_sectional_variance_four_category.{ext}",
                 dpi=300, bbox_inches="tight")
plt.show()


## 6. Select three examples with repeated threshold crossings

This section preserves the original illustrative selection method. The pooled median is computed over valid **household-years**, and crossings must be between the two clear regimes outside an exclusion band. No state probabilities are estimated here.

In [ ]:
threshold = float(valid.Y.median())
band = EXCLUSION_BAND_PP / 100.0
if not (0 < threshold - band < threshold + band < 1):
    raise ValueError("Exclusion band extends beyond valid share range")

valid["regime"] = np.select(
    [valid.Y >= threshold + band, valid.Y <= threshold - band],
    [1, -1], default=0
).astype("int8")

def count_switches(g):
    """Count clear-regime crossings within an uninterrupted household-year spell."""
    clear = g.regime.to_numpy()
    clear = clear[clear != 0]
    return int(np.sum(clear[1:] != clear[:-1])) if len(clear) > 1 else 0

spell_stats = valid.groupby("spell_id", sort=False).agg(
    hhid=("hhid", "first"), n_years=("panel_year", "size"),
    first_year=("panel_year", "min"), last_year=("panel_year", "max"),
    annual_range_pp=("Y", lambda x: float((x.max()-x.min())*100)),
)
switch_counts = valid.groupby("spell_id", sort=False)["regime"].apply(
    lambda s: int(np.sum(s.to_numpy()[s.to_numpy()!=0][1:] !=
                         s.to_numpy()[s.to_numpy()!=0][:-1]))
)
spell_stats["switches"] = switch_counts.astype(int)
longest = (spell_stats.sort_values(["n_years", "first_year"], ascending=[False, True])
           .drop_duplicates("hhid"))
eligible = longest.loc[
    (longest.n_years >= MIN_CONSECUTIVE_YEARS)
    & (longest.switches >= MIN_SWITCHES)
].copy()
screening_summary = pd.DataFrame([
    {"stage": "All households", "households": annual.hhid.nunique()},
    {"stage": f"At least one valid year (>= {MIN_ANNUAL_TRIPS} trips)", "households": valid.hhid.nunique()},
    {"stage": f"Longest consecutive spell >= {MIN_CONSECUTIVE_YEARS} years", "households": int((longest.n_years >= MIN_CONSECUTIVE_YEARS).sum())},
    {"stage": f"Also >= {MIN_SWITCHES} clear crossings", "households": len(eligible)},
])
screening_summary.to_csv(OUTPUT_DIR / "screening_counts.csv", index=False)
display(screening_summary)
print(f"Pooled median: {threshold*100:.2f}%; exclusion band: ±{EXCLUSION_BAND_PP:g} pp")
if len(eligible) < N_EXAMPLES:
    raise ValueError(f"Only {len(eligible)} illustrative households meet the rules; "
                     f"{N_EXAMPLES} required. Variance outputs above have been saved.")

# Intentional, nonrepresentative examples: maximize switches, range, and spell length.
eligible = eligible.sort_values("hhid").copy()
eligible["tie_break"] = np.random.default_rng(RANDOM_SEED).random(len(eligible))
selected = eligible.sort_values(
    ["switches", "annual_range_pp", "n_years", "tie_break"],
    ascending=[False, False, False, True]
).head(N_EXAMPLES).copy()
selected["example_label"] = [f"Household {chr(65+i)}" for i in range(N_EXAMPLES)]
examples = valid.loc[valid.spell_id.isin(selected.index)].copy()
examples["example_label"] = examples.spell_id.map(selected.example_label)
selected.to_csv(OUTPUT_DIR / "selected_households_private.csv", index_label="spell_id")
examples.to_parquet(OUTPUT_DIR / "selected_trajectories_private.parquet", index=False)
display(selected[["example_label", "n_years", "first_year", "last_year", "switches", "annual_range_pp"]])


## 7. Export three-household repeated-switching figure

The selected household trajectories are descriptive illustrations; the 2008–2017 pooled median is a shared visual reference. Blue/red shading and point colors do **not** represent inferred HMM states.

In [ ]:
plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42,
})
blue, red, gray = "#2266A5", "#CF3636", "#777777"
fig, axes = plt.subplots(1, 3, figsize=(15, 5.6), sharex=True, sharey=True)

# Shared scale: preserve zero as a reference and the same height in each panel.
max_display = min(100, max(50, int(np.ceil(examples.Y.max()*100/10))*10))
for ax, (_, item) in zip(axes.flat, selected.iterrows()):
    s = examples.loc[examples.example_label.eq(item.example_label)].sort_values("panel_year")
    x, y = s.panel_year.to_numpy(), 100*s.Y.to_numpy()
    ax.axhspan(0, threshold*100, color="#FBEAEA", alpha=0.85, zorder=0)
    ax.axhspan(threshold*100, max_display, color="#E9F3FF", alpha=0.85, zorder=0)
    ax.axhline(threshold*100, linestyle="--", color="#666666", linewidth=1, zorder=1)
    ax.plot(x, y, color=gray, linewidth=1.8, zorder=2)
    # All observations keep their true values. Ambiguous points appear gray.
    colors = np.where(s.regime.to_numpy()==1, blue,
                      np.where(s.regime.to_numpy()==-1, red, gray))
    ax.scatter(x, y, c=colors, s=45, edgecolor="white", linewidth=0.5, zorder=3)
    ax.set_title(f"{item.example_label}  (Switches = {int(item.switches)})", fontsize=12, fontweight="bold")
    ax.set_ylim(0, max_display)
    ax.set_xlim(YEAR_START-0.3, YEAR_END+0.3)
    ax.set_xticks(np.arange(YEAR_START, YEAR_END+1, 2))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.grid(axis="y", color="#DDDDDD", linewidth=0.6)
    ax.set_axisbelow(True)

fig.suptitle("Repeated Switching in Household Healthy Purchasing Behavior", fontsize=18, fontweight="bold", y=0.97)
fig.text(0.5, 0.89, "Annual healthy share of food spending (four food categories)", ha="center", fontsize=11)
fig.supxlabel("Calendar Year", fontsize=11, y=0.19)
fig.supylabel("Healthy Share of Food Spending (%)", fontsize=11, x=0.02)
legend = [
    Line2D([0],[0], color="none", marker="o", markerfacecolor=blue, markeredgecolor="white", markersize=8, label="Higher healthy purchasing"),
    Line2D([0],[0], color="none", marker="o", markerfacecolor=red, markeredgecolor="white", markersize=8, label="Lower healthy purchasing"),
    Line2D([0],[0], color="none", marker="o", markerfacecolor=gray, markeredgecolor="white", markersize=8, label="Near threshold"),
    Line2D([0],[0], color="#666666", linestyle="--", label=f"Pooled median = {threshold*100:.1f}%"),
]
fig.legend(handles=legend, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.11), fontsize=9)
fig.text(0.055, 0.025,
         f"Notes: Selected households have ≥{MIN_CONSECUTIVE_YEARS} consecutive years, ≥{MIN_ANNUAL_TRIPS} trips/year and ≥{MIN_SWITCHES} crossings "
         f"between regimes outside a ±{EXCLUSION_BAND_PP:g} pp band. Four categories include Fruits and Vegetables, "
         "cereal, snacks, and beverages. Descriptive regimes are not HMM latent states.",
         ha="left", va="bottom", fontsize=8, color="#444444", wrap=True)
fig.subplots_adjust(left=0.075, right=0.985, top=0.75, bottom=0.30, wspace=0.16)
for extension in ("png", "pdf"):
    fig.savefig(OUTPUT_DIR / f"repeated_switching_three_households_four_category.{extension}", dpi=300, bbox_inches="tight")
plt.show()

settings = {
    "category_specification": ["Fruits and Vegetables", "Cereal", "Snacks", "Beverages"],
    "healthy_spending_columns": HEALTHY_SPENDING_COLUMNS,
    "denominator": "sum of annual trip food_spend",
    "threshold_definition": "Pooled median among valid household-years",
    "median_threshold": threshold,
    "exclusion_band_pp": EXCLUSION_BAND_PP,
    "min_switches": MIN_SWITCHES,
    "min_consecutive_years": MIN_CONSECUTIVE_YEARS,
    "min_annual_trips": MIN_ANNUAL_TRIPS,
    "eligible_households": len(eligible),
    "selection": "Purposive: prioritize switch count, then range, spell length, seeded tie-break",
}
(OUTPUT_DIR / "figure_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print(f"Private figure and audit outputs: {OUTPUT_DIR}")


## 8. Output inventory and interpretation

Saved to the private four-category directory:

- `annual_four_category_spending.parquet` and `.meta.json`: full household-year spending cache.
- `variance_decomposition_four_category.csv`: exact between/within/total SS, component variances, shares, counts, overall mean.
- `variance_decomposition_four_category.png` and `.pdf`: 100% variance-decomposition chart.
- `yearly_cross_sectional_variance_four_category.csv`: year-specific cross-household moments.
- `annual_cross_sectional_variance_four_category.png` and `.pdf`: year-by-year variance chart.
- `variance_decomposition_long_panel_robustness.csv`: sensitivity for households with an >=8-year valid spell, if available.
- `screening_counts.csv`: switching example selection audit.
- `selected_households_private.csv`, `selected_trajectories_private.parquet`: sensitive illustrative household records.
- `repeated_switching_three_households_four_category.png` and `.pdf`: original-style illustrative figure.
- `figure_settings.json`: selection and measure metadata.

**Interpretation:** Within-household variance reflects annual movement (including measurement noise and transient changes), not necessarily transitions between HMM latent regimes. Same-year cross-sectional variance is distinct from the between-household SS component. 